# Part 1: URL Feature Extraction
**Multi-Layer Phishing Detection & Risk Assessment**

This notebook demonstrates the extraction of 28 numeric features across 3 categories:
1. **Length & Structure Features** (15 features)
2. **Hostname & Domain Features** (9 features)
3. **Suspicious-Pattern Features** (4 features)

In [ ]:
import sys
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.feature_extraction import extract_url_features, extract_features_from_dataset, FEATURE_COLUMNS

## 1. Feature Categories Overview
All 28 features are extracted purely through offline lexical and structural analysis without executing or fetching web content.

In [ ]:
print(f"Total features: {len(FEATURE_COLUMNS)}")
for idx, col in enumerate(FEATURE_COLUMNS, 1):
    print(f"{idx:2d}. {col}")

## 2. Test Cases on Representative URL Types

In [ ]:
test_urls = [
    ("Legitimate Standard", "https://www.google.com/search?q=machine+learning"),
    ("Phishing Lure with Keywords", "http://login.secure-update.paypal.com.attacker.biz/verify/account?id=123"),
    ("IP Address Host", "http://192.168.1.1:8080/admin/panel.php"),
    ("Scheme-less with Subdomains", "sub1.sub2.example.co.uk/checkout/pay"),
    ("At-Symbol Authority Injection", "http://legit-site.com@phishing-target.com/login"),
    ("URL Shortener", "https://bit.ly/3xYz123"),
]

records = []
for label_desc, u in test_urls:
    feats = extract_url_features(u)
    feats["_type"] = label_desc
    feats["_url"] = u
    records.append(feats)

sample_features_df = pd.DataFrame(records).set_index(["_type", "_url"])
sample_features_df[["url_length", "has_ip_address", "is_https", "has_suspicious_keyword", "has_url_shortener", "num_subdomains"]]

## 3. Batch Extraction on Cleaned Dataset Samples

In [ ]:
cleaned_csv = project_root / "dataset" / "processed" / "cleaned_dataset.csv"
if cleaned_csv.exists():
    cleaned_df = pd.read_csv(cleaned_csv)
    sample_df = cleaned_df.head(100)
    feats_sample = extract_features_from_dataset(sample_df, url_col='url', batch_size=50, verbose=True)
    print("\nSample extracted features matrix:")
    print(feats_sample.head())
else:
    print("Cleaned dataset not found. Run notebooks/data_preprocessing.ipynb or main.py first.")